In [12]:
import pandas as pd
from statsmodels.tsa.ardl import ardl_select_order

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

df = df[["cpi_headline", "cpi_transport"]]
df = df.asfreq("ME")

df.head()


,cpi_headline,cpi_transport
date,,
2020-01-31,122.4,115.0
2020-02-29,122.4,113.9
2020-03-31,120.9,104.0
2020-04-30,117.6,90.0
2020-05-31,117.9,90.9


In [13]:
from statsmodels.tsa.stattools import adfuller

print("ADF p-value (CPI headline):", adfuller(df["cpi_headline"])[1])


ADF p-value (CPI headline): 0.9008444051208608


In [14]:
sel = ardl_select_order(
    endog=df["cpi_headline"],
    exog=df[["cpi_transport"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print(sel.model)


In [15]:
ardl_ch = sel.model.fit()
print(ardl_ch.summary())


                              ARDL Model Results                              
Dep. Variable:           cpi_headline   No. Observations:                   71
Model:                     ARDL(1, 1)   Log Likelihood                   6.653
Method:               Conditional MLE   S.D. of innovations              0.220
Date:                Thu, 08 Jan 2026   AIC                             -3.305
Time:                        09:06:22   BIC                              7.937
Sample:                    02-29-2020   HQIC                             1.160
                         - 11-30-2025                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.5157      0.753      0.685      0.496      -0.988       2.019
cpi_headline.L1      0.9754      0.011     90.028      0.000       0.954       0.997
cpi_transport.L0     0.2150 

In [16]:
df2 = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

df2 = df2[["cpi_headline", "ron97", "diesel"]]
df2 = df2.asfreq("ME")

sel2 = ardl_select_order(
    endog=df2["cpi_headline"],
    exog=df2[["ron97", "diesel"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

ardl_ch2 = sel2.model.fit()
print(ardl_ch2.summary())


                              ARDL Model Results                              
Dep. Variable:           cpi_headline   No. Observations:                   71
Model:                     ARDL(4, 1)   Log Likelihood                  -0.400
Method:               Conditional MLE   S.D. of innovations              0.243
Date:                Thu, 08 Jan 2026   AIC                             16.800
Time:                        09:06:22   BIC                             34.437
Sample:                    05-31-2020   HQIC                            23.779
                         - 11-30-2025                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               2.8796      0.936      3.075      0.003       1.006       4.753
cpi_headline.L1     1.0919      0.072     15.229      0.000       0.949       1.235
cpi_headline.L2    -0.2472      

In [17]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan


In [18]:
def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}.")

    # align exog to residual sample
    X0 = X0_full[-n_u:, :]

    # auxiliary regression: u_t on X + lagged u
    U_lags = lagmat(u, maxlag=nlags, trim="both")
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Non-finite values (NaN/inf) in BG auxiliary regression inputs.")

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(X0_full.shape[0]),
    }


def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]  # align
    X0 = sm.add_constant(X0, has_constant="add")  # force constant

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1]),
    }


In [21]:
res = ardl_ch   # ARDL(1,1) with cpi_transport

bg_out = bg_test_ardl_aligned(res, nlags=2)
bp_out = bp_test_ardl(res)

bg_out, bp_out


({'LM stat': 0.5497538622468618,
  'LM p-value': 0.7596656086653031,
  'F stat': 0.248414345377076,
  'F p-value': 0.7807738187594583,
  'nobs_aux': 68,
  'resid_len': 70,
  'exog_rows_full': 71},
 {'LM stat': 1.683374201479384,
  'LM p-value': 0.19447670319447097,
  'F stat': 1.6755722982893158,
  'F p-value': 0.1998924143613866,
  'k_exog': 2})

In [22]:
import joblib
joblib.dump(res, "../data/joblib/ardl_cpi_headline_brent.joblib")

['../data/joblib/ardl_cpi_headline_brent.joblib']

In [23]:
res = ardl_ch2  # ARDL(4,1) with ron97 (diesel got dropped by selection)

bg_out2 = bg_test_ardl_aligned(res, nlags=2)
bp_out2 = bp_test_ardl(res)

bg_out2, bp_out2


({'LM stat': 0.8771047255574899,
  'LM p-value': 0.6449694273425014,
  'F stat': 0.4117004705659418,
  'F p-value': 0.6643497202586705,
  'nobs_aux': 65,
  'resid_len': 67,
  'exog_rows_full': 71},
 {'LM stat': 3.9221613215487827,
  'LM p-value': 0.14070628297037654,
  'F stat': 1.989750519661317,
  'F p-value': 0.14509887475294778,
  'k_exog': 3})

In [24]:
import joblib
joblib.dump(res, "../data/joblib/ardl_cpi_headline_brent_fuel_alt.joblib")

['../data/joblib/ardl_cpi_headline_brent_fuel_alt.joblib']